In [1]:
import os, torch

root_path = os.path.dirname(os.getcwd())
os.chdir(root_path)
from modules.utils.general import get_device

### Fetch Data

In [ ]:
from modules.data_pipeline.retrieval import BaseDatasetRetriever
retriever = BaseDatasetRetriever(dataset_name="svo-probes", data_root=root_path+'/data')

### Convert to Diagrams

In [ ]:
from modules.symbolic import text_processor
functor = text_processor.TextProcessor('/Users/tls/Desktop/Work/COMP0267/assignment_5/COMP0267_CW/bobcat')
functor.text2diagram(path=root_path+'/data/svo-probes/processed', dataset=retriever.data, text_labels=retriever.text_labels)

### Compile to Quantum Circuits

In [2]:
from modules.utils.general import load_pkl, store_pkl
from modules.utils.general import get_device
from modules.compilation.classical.tensor_network import TNCompiler
from modules.compilation.classical.decompositions import MPS

obmap = {'n': 16, 's': 16, 'p': 16, 'out': 512}
mps_proc = MPS(bond_dim=32, max_order=2)
ansatz = TNCompiler(obmap=obmap, decomp_fn=mps_proc)
compile_id = type(ansatz).__name__ + '_' + str(obmap['n']) + '_' + str(obmap['s']) + '_' + str(obmap['p']) + '_' + str(obmap['out'])

In [3]:
path = os.path.join(root_path, f'data/svo-probes/processed/compiled/{compile_id}')
if not os.path.exists(path): os.mkdir(path)

splits = ['train', 'val', 'test', 'swap']
for split in splits:
    df = load_pkl(f'data/svo-probes/processed/{split}.pkl')
    compiled_df = ansatz.compile_dataset(df)
    store_pkl(compiled_df, os.path.join(path, f'{split}.pkl'))
    print(f"Compiled {split} dataset saved to {os.path.join(path, f'{split}.pkl')}")

--> Found target diagram columns: ['corrected_sentence_diagram']
--> Total rows to process: 5267


100%|██████████| 5267/5267 [00:00<00:00, 40054.46it/s]

Compiled train dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/TNCompiler_16_16_16_512/train.pkl
--> Found target diagram columns: ['corrected_sentence_diagram']
--> Total rows to process: 1829



100%|██████████| 1829/1829 [00:00<00:00, 22258.41it/s]

Compiled val dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/TNCompiler_16_16_16_512/val.pkl
--> Found target diagram columns: ['corrected_sentence_diagram']
--> Total rows to process: 1833



100%|██████████| 1833/1833 [00:00<00:00, 85407.86it/s]


Compiled test dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/TNCompiler_16_16_16_512/test.pkl
--> Found target diagram columns: ['corrected_sentence_diagram']
--> Total rows to process: 95


100%|██████████| 95/95 [00:00<00:00, 108571.90it/s]


Compiled swap dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/TNCompiler_16_16_16_512/swap.pkl


### Load Towers

In [3]:
train_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/train.pkl')
val_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/val.pkl')
test_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/test.pkl')
swap_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/swap.pkl')

In [4]:
from modules.models.text.einsum_quantum import QCModel
from modules.models.vision.image_model import TTNImageModel
from modules.models.text.einsum_classical import TNModel

BATCH_SIZE = 256
DEV = 'cpu' # get_device()

image_model = TTNImageModel(embedding_dim=512)
image_model.to(DEV)
image_model.compile_batch(batch_size=BATCH_SIZE)

text_model = TNModel(out_dim=512)
text_model.to(DEV)
txt_stream = train_df['corrected_sentence_symbols'].tolist() + val_df['corrected_sentence_symbols'].tolist() + test_df['corrected_sentence_symbols'].tolist() + swap_df['corrected_sentence_symbols'].tolist()
text_model.from_symbols(txt_stream)

100%|██████████| 9024/9024 [00:00<00:00, 379701.44it/s]


In [5]:
from modules.data_pipeline.datasets import SVODataset, svo_collate_fn
from torch.utils.data import DataLoader
from torchvision.transforms import v2
# , num_workers=4, pin_memory=True
IMG_TRANSFORM = v2.Compose([v2.ToImage(),               
                            v2.ToDtype(torch.float32, scale=True),
                            v2.Resize((64, 64))])

train_dataset = SVODataset(train_df, 'data/svo-probes/raw/images.zip', image_transform=IMG_TRANSFORM)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn, shuffle=True)

val_dataset = SVODataset(val_df, 'data/svo-probes/raw/images.zip', image_transform=IMG_TRANSFORM)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn)

test_dataset = SVODataset(test_df, 'data/svo-probes/raw/images.zip', image_transform=IMG_TRANSFORM)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn)

swap_dataset = SVODataset(swap_df, 'data/svo-probes/raw/images.zip', image_transform=IMG_TRANSFORM)
swap_loader = DataLoader(swap_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn)

### Learn and Evaluate

In [6]:
from modules.models.fusion.criteria import InfoNCE
from modules.models.fusion.engine import ContrastiveTrainer, MMEvaluator

LR = 1e-3
optimizer = torch.optim.Adam(list(text_model.parameters()) + list(image_model.parameters()), lr=LR)
loss_fn = InfoNCE()

trainer = ContrastiveTrainer(image_model, text_model, optimizer, loss_fn, DEV)
evaluator = MMEvaluator(image_model, text_model, DEV)

In [7]:
import mlflow, time
from modules.utils.general import set_seed
from modules.models.fusion.engine import svo_mapper
from pathlib import Path

EPOCHS = 25
SEED = int.from_bytes(os.urandom(4))
set_seed(SEED)

exp_name = 'svo-probes'

run_name = f"{int(time.time())}"
checkpoint_path = f"./checkpoints/{exp_name}/{run_name}.pt"
Path(os.path.dirname(checkpoint_path)).mkdir(parents=True, exist_ok=True)
mlf_path = os.path.join(root_path, f'mlf_dbs/{exp_name}.db')


mlflow.pytorch.autolog()
mlflow.set_tracking_uri(f"sqlite:///{mlf_path}")
mlflow.set_experiment(exp_name)

with mlflow.start_run(run_name=run_name):
    mlflow.log_params({
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "temperature_parameter": loss_fn.temperature,
        "device_target": str(DEV),
        "seed": SEED
        })
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        loss = trainer.train_epoch(train_loader, svo_mapper)
        end_time = time.time()
        
        svo_acc = evaluator.evaluate_image_choice(val_loader)
        
        mlflow.log_metric("train_loss", loss, step=epoch)
        mlflow.log_metric("svo_acc", svo_acc, step=epoch)
        print(f"Epoch {epoch} | Loss: {loss:.4f} | SVO: {svo_acc:.4f} | Time: {end_time - start_time:.2f}s")

        torch.save({
            "image": image_model.state_dict(),
            "text": text_model.state_dict(),
            "epoch": epoch,
            "train_loss": loss,
            "val_lacc": svo_acc if val_loader else None
            }, checkpoint_path)

Epoch 0 | Loss: 5.8977 | SVO: 0.4975 | Time: 108.43s
Epoch 1 | Loss: 5.8371 | SVO: 0.4828 | Time: 111.01s
Epoch 2 | Loss: 5.6913 | SVO: 0.4992 | Time: 108.72s
Epoch 3 | Loss: 5.6312 | SVO: 0.5085 | Time: 107.71s
Epoch 4 | Loss: 5.5546 | SVO: 0.5041 | Time: 115.30s
Epoch 5 | Loss: 5.4486 | SVO: 0.5243 | Time: 106.81s
Epoch 6 | Loss: 5.3133 | SVO: 0.5254 | Time: 119.77s
Epoch 7 | Loss: 5.1541 | SVO: 0.5243 | Time: 128.17s
Epoch 8 | Loss: 4.9628 | SVO: 0.5850 | Time: 128.25s
Epoch 9 | Loss: 4.6747 | SVO: 0.5888 | Time: 128.53s
Epoch 10 | Loss: 4.4114 | SVO: 0.6249 | Time: 129.01s
Epoch 11 | Loss: 4.0650 | SVO: 0.6326 | Time: 138.58s
Epoch 12 | Loss: 3.7500 | SVO: 0.6523 | Time: 132.70s
Epoch 13 | Loss: 3.4315 | SVO: 0.6484 | Time: 129.33s
Epoch 14 | Loss: 3.1667 | SVO: 0.6594 | Time: 129.57s
Epoch 15 | Loss: 2.8991 | SVO: 0.6632 | Time: 136.25s
Epoch 16 | Loss: 2.7005 | SVO: 0.6714 | Time: 129.04s
Epoch 17 | Loss: 2.5202 | SVO: 0.6851 | Time: 128.41s
Epoch 18 | Loss: 2.3383 | SVO: 0.6796 

In [ ]:
from itertools import count

max_char = max(set(['a','b','c']))
counter = count(start=ord(max_char) + 1)
chr(next(counter))